# **walter**

## **Project Setup**

In [5]:
import sys
import os
from pathlib import Path
import subprocess

import torch
import pandas as pd
from huggingface_hub import HfApi

def get_env_or_prompt(env_name: str, prompt_name: str):
    token = os.getenv(env_name)
    if token:
        return token

    import getpass
    return getpass.getpass(f"{prompt_name}: ")


def get_github_token():
    return get_env_or_prompt("GITHUB_TOKEN", "GitHub token")


def get_hf_token():
    return get_env_or_prompt("HF_TOKEN", "Hugging Face token")


IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "walter"
GIT_BRANCH = "main"


if IN_COLAB:
    print("Colab detected. Setting up repository...")

    %cd /content

    github_token = get_github_token()
    repo_url = f"https://{github_token}@github.com/Mango-Cats/{REPO_NAME}.git"

    if Path(REPO_NAME).exists():
        %cd {REPO_NAME}
        !git fetch origin
        !git reset --hard origin/{GIT_BRANCH}
    else:
        !git clone {repo_url}
        %cd {REPO_NAME}

    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)


hf_token = get_hf_token()
api = HfApi(token=hf_token)

print("HF authenticated user:")
print(api.whoami())


project_root = Path("/content") / REPO_NAME
os.chdir(project_root)

print("Project root:", project_root)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch version:", torch.__version__)
print("Device:", DEVICE)

Colab detected. Setting up repository...
/content
Cloning into 'walter'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 134 (delta 64), reused 94 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 117.21 KiB | 14.65 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/walter
HF authenticated user:
{'type': 'user', 'id': '68b91f73b81aa6dfbd75984b', 'name': 'zrygan', 'fullname': 'Zhean Robby Ganituen', 'email': 'zhean_robby_ganituen@dlsu.edu.ph', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1780272000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/Fp4ErHoZ_gN2BtQvG9Iff.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'walter', 'role': 'read', 'createdAt': '2026-05-02T02:47:29.965Z'}}}
Project root: /content/walter
Torch version: 2.10.0+cu128
De

## **Nomenclature and Terminologies**

The dataset $\mathcal{D}_{\text{raw}}$ (represented as `D_raw` in the source code) refers to the raw Philippine human-drug registry, which is freely available as a `.csv` file at [https://verification.fda.gov.ph/drug_productslist.php](https://verification.fda.gov.ph/drug_productslist.php).

The intermediate dataset, $\mathcal{D}_{\text{clean}}$ (`D_clean`), is the result of passing $\mathcal{D}_{\text{raw}}$ through the preprocessing pipeline. 

The final dataset, $\mathcal{D}_{\text{train}}$ (`D_train`), is used to train a weighted sum of similarity measures via a genetic algorithm. It consists of ordered pairs of drugs formed from the cleaned registry, such that every pair $(x, y) \in \mathcal{D}_{\text{clean}} \times \mathcal{D}_{\text{clean}}$. This training dataset is partitioned into two disjoint subsets:

* **$P \subset \mathcal{D}_{\text{train}}$** (`P`) is the set of known positives, consisting of ordered drug pairs that are manually verified as LASA.
* **$U \subset \mathcal{D}_{\text{train}}$** (`U`) is the unlabeled noise set, consisting of randomly paired drugs from $\mathcal{D}_{\text{clean}}$. $U$ acts as the noise class (its true labels are unknown), so it may contain undetected LASA pairs.

Furthermore, $|U| \gg |P|$, $P \cap U = \emptyset$, and $P \cup U = \mathcal{D}_{\text{train}}$.

## **Preprocessing**

The first step is to preprocess (load, validate, clean) the FDA human drug registry dataset to construct our $\mathcal{D}_\text{clean}$ dataset.

Ensure that the FDA human drug registry dataset exists anywhere starting from the root folder and has the same filename defined by `PRIMARY_FNAME`.

The code for this section is located at [`/src/preprocessing.py`](/src/preprocessing.py).

The function `master_maker` is the coordinator function that performs data loading, validation, cleaning, and reporting. 

In [6]:
import src.preprocessing as pre
D_clean = pre.master_maker(sort=True, save=True)

ModuleNotFoundError: No module named 'src.preprocessing'

Let's look at the info of the dataset.

In [ ]:
D_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 22838 entries, 0 to 22837
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Brand Name  22838 non-null  str  
dtypes: str(1)
memory usage: 178.6 KB


Then, the head of the dataset.

In [ ]:
D_clean.head()

,Brand Name
0,0.9% NaCl-Sapher
1,0.9% Sodchlorsaph
2,1 Ceeplus
3,1000Vc
4,2-Gen


Finally, let's look at a slice of 10 entries in the dataset by using the `get_rand_entries()` function.

In [ ]:
display(pre.get_rand_entries(df=D_clean, count=10))

,Brand Name
19292,Supremus
19293,Suprezol
19294,Suprine
19295,Suprinex
19296,Suprivin
19297,Supzine 10
19298,Suquin
19299,Surcon
19300,Sureboost C Plus
19301,Sureboost Daily


## **True LASA Pairs**

Now that we have the $\mathcal{D}_\text{clean}$ we can now proceed with constructing $\mathcal{D}_\text{train}$. We will prioritize constructing the subset of true LASA pairs, or the set $P$.

The code for this section is located at [`/src/proposer/`](/src/proposer/).

### **HuggingFace Models**

In [ ]:
from src.proposer.hf import HFModel, response
from src.preprocessing import table_to_string

model = HFModel.TINY_LLAMA

iters = 1
P_hf = []

for _ in range(iters):
    sample_df = D_clean.sample(n=1)
    remaining_df = D_clean.drop(sample_df.index)

    sample_str = table_to_string(sample_df)
    remaining_str = table_to_string(remaining_df)

    proposals = response(sample_str, model=model)

    P_hf.append((sample_str, proposals))

print(P_hf)

<walter> Proposing LASA for: Ateron.


## **Noise Pairs**

Now that we have $P$, we can now complete constructing $\mathcal{D}_\text{train}$ by constructing the set $U$ or the unlabeled noise set.

The code for this section is located at [`/src/noise.py`](/src/noise.py)

In [ ]:
import src.noise as noise
from src.utils import finder

sample_file: Path = finder(fname="sample_true_lasa.csv")
sample_P: pd.DataFrame = pd.read_csv(filepath_or_buffer=sample_file)

lasa_set = noise.get_lasa_set(true_df=sample_P)

assert len(lasa_set) == 20

sample_U = noise.make_noise(fda_df=D_clean, true_df=sample_P, n=3)

sample_U

,Drug Name 1,Drug Name 2
0,Vendicom,Tgxime
1,Vendicom,Verlene-20
2,Tgxime,Vendicom
3,Tgxime,Verlene-20
4,Verlene-20,Vendicom
5,Verlene-20,Tgxime


## **Assembling**

Now that both subsets are complete. Assembling $\mathcal{D}_\text{train}$ is simply a concatenation of $P$ and $U$. 

In [ ]:
...

Ellipsis